In [89]:
import ollama
import pandas as pd
from openai import OpenAI

# query parser

In [90]:
api_key = ""
client = OpenAI(api_key=api_key)

In [100]:
query_parser_knowledge_base = """

You are an intelligent text summariser for a recipe recommendation system.

Your task is to convert a user's natural language into keywords

Extract or develop only the relevant keywords according to the text:

type of meal for example breakfast, lunch, dinner, snack, dessert, drink, full course (infer from time/context if not explicit)
time required (if mentioned or implied like quick, fast
dietary constraints for e.g., no eggs, no dairy, vegan, no onion, etc.
ingredients to include
ingredients to avoid
cuisine (e.g., Indian, Italian, Greek, etc.)
dish type (e.g., curry, soup, salad, rice, pasta, etc.)
nutrition e.g., high protein, lowcalorie, healthy, light)
difficulty for example easy, medium, hard

Please be aware of the context, do not add unnecessary and irrelevant keywords

Rules:
- Infer intelligently from context (e.g., "7pm" → dinner, "after gym" → high protein)
- Do NOT explain anything

The response should be between atleast 3 and maximum 8 keywords strictly in a single phrase
Please double check the response format
"""

In [113]:
def query_parser(client_query):
    response = ollama.chat(
        model="qwen3.5:9b",
        think=False,
        messages=[
            {
                "role": "system",
                "content": f"""
    
    {query_parser_knowledge_base}
    """
            },
            {
                "role": "user",
                "content": f"""
    Develop keywords from this text.
    
    Text:
    {client_query}
    
    """
            }
        ]
    )
    
    # print(response['message']['content'])
    return response['message']['content']

In [114]:
# c_query = "I am hungry and it is 7pm and I got back from the gym. Suggest something quick"
# c_query = "had a good long morning run. suggest something to drink"
# c_query = "guests coming over for dinner"
# c_query = "feeling like eating something spicy"
c_query = "I’m planning a vegetarian dinner for about 8–10 people, can you suggest a menu with a good variety of dishes including something filling, something light, and maybe a side or accompaniment?"
query_parser(c_query)

'vegetarian dinner large group filling light side'

In [105]:
query_parser_df = pd.read_csv("/home/linux/linux/ProjectsReal/chef/query_parser_long.csv")

In [106]:
query_parser_df.head()

,user input
0,I just got back home from the gym and it's alr...
1,It's been a long day at work and I don't have ...
2,I woke up late today and don't have much time ...
3,I skipped lunch and now I'm starving in the ev...
4,I just came back from a walk and want to eat s...


In [107]:
query_parser_df['qwen0.8B'] = query_parser_df['user input'].apply(query_parser)

In [108]:
query_parser_df.head()

,user input,qwen0.8B
0,I just got back home from the gym and it's alr...,"dinner meal, dinner ready, high protein, quick..."
1,It's been a long day at work and I don't have ...,vegetarian dinner time required short meal dis...
2,I woke up late today and don't have much time ...,quick vegetarian lunch light protein high protein
3,I skipped lunch and now I'm starving in the ev...,"dinner, light, low-fat, full course"
4,I just came back from a walk and want to eat s...,light meal easy digestion dinner suitable


In [111]:
query_parser_df['qwen2B'] = query_parser_df['user input'].apply(query_parser)

In [112]:
query_parser_df.head()

,user input,qwen0.8B,qwen2B
0,I just got back home from the gym and it's alr...,"dinner meal, dinner ready, high protein, quick...","dinner, high protein, quick, healthy"
1,It's been a long day at work and I don't have ...,vegetarian dinner time required short meal dis...,vegetarian dinner quick easy
2,I woke up late today and don't have much time ...,quick vegetarian lunch light protein high protein,vegetarian quick vegetarian breakfast light fu...
3,I skipped lunch and now I'm starving in the ev...,"dinner, light, low-fat, full course",dinner meal hungry filling light quick healthy...
4,I just came back from a walk and want to eat s...,light meal easy digestion dinner suitable,light dinner healthy low calorie easy digest


In [115]:
query_parser_df['qwen9B'] = query_parser_df['user input'].apply(query_parser)

In [116]:
query_parser_df.head()

,user input,qwen0.8B,qwen2B,qwen9B
0,I just got back home from the gym and it's alr...,"dinner meal, dinner ready, high protein, quick...","dinner, high protein, quick, healthy","dinner, quick, high protein"
1,It's been a long day at work and I don't have ...,vegetarian dinner time required short meal dis...,vegetarian dinner quick easy,"dinner, quick, easy, vegetarian, simple"
2,I woke up late today and don't have much time ...,quick vegetarian lunch light protein high protein,vegetarian quick vegetarian breakfast light fu...,"breakfast, quick, vegetarian, light, keeps me ..."
3,I skipped lunch and now I'm starving in the ev...,"dinner, light, low-fat, full course",dinner meal hungry filling light quick healthy...,"dinner, filling, light"
4,I just came back from a walk and want to eat s...,light meal easy digestion dinner suitable,light dinner healthy low calorie easy digest,"dinner, light, easy to digest, satisfying, pos..."


In [117]:
def query_parser_openai(client_query):
    response = client.chat.completions.create(
        model="gpt-4.1-mini",  # fast + cheap + good for extraction
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": query_parser_knowledge_base
            },
            {
                "role": "user",
                "content": f"Develop keywords from this text. Text:\n{client_query}"
            }
        ]
    )

    result = response.choices[0].message.content
    return result
    # if result:
        # return json.loads(result)
    # return None

In [118]:
query_parser_df['gpt-4.1-mini'] = query_parser_df['user input'].apply(query_parser_openai)

In [119]:
query_parser_df.head()

,user input,qwen0.8B,qwen2B,qwen9B,gpt-4.1-mini
0,I just got back home from the gym and it's alr...,"dinner meal, dinner ready, high protein, quick...","dinner, high protein, quick, healthy","dinner, quick, high protein","dinner, quick, high protein, easy, post-gym"
1,It's been a long day at work and I don't have ...,vegetarian dinner time required short meal dis...,vegetarian dinner quick easy,"dinner, quick, easy, vegetarian, simple","dinner, vegetarian, simple, comforting, quick,..."
2,I woke up late today and don't have much time ...,quick vegetarian lunch light protein high protein,vegetarian quick vegetarian breakfast light fu...,"breakfast, quick, vegetarian, light, keeps me ...","breakfast, quick, vegetarian, light, filling, ..."
3,I skipped lunch and now I'm starving in the ev...,"dinner, light, low-fat, full course",dinner meal hungry filling light quick healthy...,"dinner, filling, light","dinner, filling, light, easy, quick"
4,I just came back from a walk and want to eat s...,light meal easy digestion dinner suitable,light dinner healthy low calorie easy digest,"dinner, light, easy to digest, satisfying, pos...","dinner, light, easy to digest, satisfying, easy"


In [120]:
query_parser_df.to_csv("/home/linux/linux/ProjectsReal/chef/query_parser_long.csv", index=False)

In [ ]:
original_user_query = 
converted_keywords = 

In [124]:
def chat_with_recipe(recipe, user_query, chat_history=None):
    if chat_history is None:
        chat_history = []

    system_prompt = f"""
You are a helpful vegetarian cooking assistant.

You will be given a recipe. The user will ask questions about it.

You can:
- explain steps
- simplify instructions
- suggest substitutions
- adjust quantities
- estimate time
- make it healthier or quicker

Always base your answers ONLY on the given recipe.
Be clear, concise, and conversational.

Recipe:
{recipe}
"""

    messages = [
        {"role": "system", "content": system_prompt}
    ]

    # Add previous conversation
    messages.extend(chat_history)

    # Add current user query
    messages.append({"role": "user", "content": user_query})

    response = ollama.chat(
        model="qwen3.5:2b",
        messages=messages,
        think=False,
        stream=True
    )

    assistant_reply = response['message']['content']

    # Update chat history
    chat_history.append({"role": "user", "content": user_query})
    chat_history.append({"role": "assistant", "content": assistant_reply})

    return assistant_reply, chat_history

In [122]:
recep = {'id': 808, 
			'recepie name': 'Hurry Up Hero', 
			'ingredients': [
					'1⁄2 piece whole-wheat pita pocket bread', 
				   '1⁄2 teaspoon yellow mustard', 
				   '1 slice pepper jack cheese or nondairy cheese', 
				   '3 thin slices green pepper', 
				   '3 slices cucumber', 
				   '2 thin slices onion', 
				   '2 slices tomato', 
				   '1⁄4 avocado, peeled, pitted, and sliced thin', 
				   '1⁄4 cup shredded lettuce'], 
			'directions': [
					'Open the pocket in the pita bread and spread the mustard inside the pocket.', 
					'Place the cheese, green pepper, cucumber, onion, tomato, avocado, and lettuce inside the pocket.'], 
			'prep time': '5 min', 
			'cook time': '0 min', 
			'yield': '1 serving', 
			'per serving': 'calories 285 (from fat 147); fat 16g (saturated 7g); cholesterol 30mg; sodium 347mg; carbohydrate 27g (dietary fiber 7g); protein 11g.', 
			'vary it': 'Use mayo if you don’t have mustard, or drizzle a little olive oil and vinegar over the filling. '
						'If you’re out of tomatoes, medium salsa works great as a substitute. '
						'In a pinch, you can use cooked asparagus spears or tiny broccoli florets if you’re avocado-less. '
						'And you can use spinach in place of the lettuce.', 
			'tip': '', 
			'note': '', 
			'tags': ['lunch', 'quick', 'vegetarian', 'healthy', 'snack', 'whole wheat', 'no cook', 'gluten']}

In [125]:
chat_with_recipe(recep, "Can I change some ingredients?")

('Absolutely! Here are your available substitutions:\n\n*   **Mustard**: Use mayonnaise or drizzle a little olive oil and vinegar over the filling.\n*   **Tomato**: Medium salsa is great if you need more texture without tomatoes, though keep in mind this changes the flavor profile slightly.\n*   **Avocado**: If you\'re out of avocados, you can use cooked asparagus spears or tiny broccoli florets inside the pocket.\n*   **Lettuce**: You can replace the shredded lettuce with spinach inside the pocket.\n\nYou can swap many of these ingredients without affecting the "no cook" nature of your dish!',
 [{'role': 'user', 'content': 'Can I change some ingredients?'},
  {'role': 'assistant',
   'content': 'Absolutely! Here are your available substitutions:\n\n*   **Mustard**: Use mayonnaise or drizzle a little olive oil and vinegar over the filling.\n*   **Tomato**: Medium salsa is great if you need more texture without tomatoes, though keep in mind this changes the flavor profile slightly.\n*  

In [144]:
main_prompt = f"""You are a helpful vegetarian cooking assistant.

You will be given a recipe. The user will ask questions about it.

You can:
- explain steps
- simplify instructions
- suggest substitutions
- adjust quantities
- estimate time
- make it healthier or quicker

Always base your answers ONLY on the given recipe.
Be clear, concise, and conversational.

Stick to the above recipe only, do not diverge a lot if asked for variations or substitutions
Do not hesitate to say that it may not be possible to make the dish if major ingredients are not available with the user

Strictly keep your answers short, with 1 to 3 sentences, unless asked for detailed explaination


Recipe:
{recep}
"""



In [147]:
messages = [
    {"role": "system", 
     "content": main_prompt}]

while True:
    user_input = input("You: ")

    if user_input == 'exit':
        break

    
    
    messages.append({"role": "user", "content": user_input})

    stream = ollama.chat(
        model="qwen3.5:9b",
        messages=messages,
        stream=True,
        think=False
    )

    print("Assistant:", end=" ", flush=True)

    full_reply = ""
    for chunk in stream:
        content = chunk["message"]["content"]
        print(content, end="", flush=True)
        full_reply += content

    print()

    messages.append({"role": "assistant", "content": full_reply})

You:  i do not have a pita bread, or any other bread, tortillas, baguette. what to do?


Assistant: Based on the "Hurry Up Hero" recipe provided:

I cannot suggest substitutions for the bread itself (like using lettuce wraps or different grains), as those would change the dish into something completely different.

Since the recipe relies on a pocket to hold the fillings, and the notes don't list an alternative bread base, **you cannot make this specific "Hurry Up Hero" sandwich** without some form of pocket bread (like a pita) or a similar vessel (like a sturdy tortilla or bun). The recipe specifically calls for opening a pocket in the pita.

If you are strictly sticking to only the ingredients and steps listed in *this* recipe, this specific dish isn't possible right now. You might need to wait until you have pita bread available.


You:  do you have any other suggestions?


Assistant: Based **only** on the provided "Hurry Up Hero" recipe and its specific "Vary it" notes, there are no alternative base suggestions listed (such as tortillas or buns) because the recipe requires a pocket to hold the mustard and fillings. The note explicitly states you can use mayo instead of mustard or salsa for tomatoes, but it does not offer a substitute for the bread itself.

Therefore, strictly sticking to this recipe's constraints, if you lack pita, tortillas, baguettes, or any other bread/bun capable of holding a pocket, you cannot make this specific dish at this time.


You:  what was the first question that i asked you with respect to this conversation?


...

You:  But I can just me a salad instead right? Just mix the vegetables, add the sauce and eat it as it is


Assistant: Yes, technically you can turn those ingredients into a salad or bowl since they are listed as "vegetables," but strictly following the **"Hurry Up Hero" recipe steps**, you cannot do this.

The directions explicitly state: *"Open the **pocket in the pita bread** and spread the mustard inside..."* then *"Place the... ingredients **inside the pocket**."*

If you eat the vegetables loose with sauce, you are following the ingredient list but ignoring the required **step** of using the bread pocket. So, while possible to eat those items, it wouldn't be making the "Hurry Up Hero" dish as defined by this specific recipe's instructions.


You:  that is fine. i will eat it as it is without trying to make an accurate dish


Assistant: That sounds like a great plan! You can definitely mix up those vegetables, add the mustard (or mayo if you prefer), and enjoy a healthy bowl of "Hurry Up Hero" ingredients right now. It's a quick, nutritious lunch even without the pocket bread. Enjoy your veggie medley!


You:  exit


In [148]:
messages

[{'role': 'system',
  'content': "You are a helpful vegetarian cooking assistant.\n\nYou will be given a recipe. The user will ask questions about it.\n\nYou can:\n- explain steps\n- simplify instructions\n- suggest substitutions\n- adjust quantities\n- estimate time\n- make it healthier or quicker\n\nAlways base your answers ONLY on the given recipe.\nBe clear, concise, and conversational.\n\nStick to the above recipe only, keep your responses short, and do not diverge a lot if asked for variations or substitutions\nDo not hesitate to say that it may not be possible to make the dish if major ingredients are not available with the user\n\nRecipe:\n{'id': 808, 'recepie name': 'Hurry Up Hero', 'ingredients': ['1⁄2 piece whole-wheat pita pocket bread', '1⁄2 teaspoon yellow mustard', '1 slice pepper jack cheese or nondairy cheese', '3 thin slices green pepper', '3 slices cucumber', '2 thin slices onion', '2 slices tomato', '1⁄4 avocado, peeled, pitted, and sliced thin', '1⁄4 cup shredded l

In [ ]:
i do not have a pita bread, or any other bread, tortillas, baguette. what to do?

do you have someother suggestion?

In [150]:
available_choices = [{'id': 772, 'recepie name': 'Morning Rice Salad', 'ingredients': ['1 container (1 cup) instant, ready-to-serve brown rice', '1/4 teaspoon cinnamon', '1 apple, cored and diced', '1/2 navel orange, diced', '1/2 cup dried cranberries', '1/4 cup chopped dates', '1/4 cup shredded sweetened coconut', '1/2 cup low-fat vanilla yogurt or soy yogurt'], 'prep time': 1, 'cook time': 1},
{'id': 799, 'recepie name': 'Eggs a la King', 'ingredients': ['1 tablespoon butter', '2 tablespoons chopped green pepper', '2 tablespoons chopped onion', '1/4 cup sliced mushrooms', '2 slices whole-grain bread', '3/4 cup low-fat cream of celery or cream of mushroom soup', '1/2 cup shredded low-fat American or cheddar cheese', '1/4 cup milk', '2 hard-boiled eggs, sliced'], 'prep time': 4, 'cook time': 4}]

In [156]:
input_choice = int(input("Write the ID of your choice : "))

Write the ID of your choice :  772


In [157]:
type(input_choice)

int

In [158]:
selected_recipe = next((item for item in available_choices if item.get("id") == input_choice), None)

In [159]:
print(selected_recipe)

{'id': 772, 'recepie name': 'Morning Rice Salad', 'ingredients': ['1 container (1 cup) instant, ready-to-serve brown rice', '1/4 teaspoon cinnamon', '1 apple, cored and diced', '1/2 navel orange, diced', '1/2 cup dried cranberries', '1/4 cup chopped dates', '1/4 cup shredded sweetened coconut', '1/2 cup low-fat vanilla yogurt or soy yogurt'], 'prep time': 1, 'cook time': 1}
